# Урок 2.4. Переход на Qdrant: Docker и продакшен-класс

**Модуль 2 «Embeddings и векторные базы данных» · Курс «LLM для Python-разработчиков»**

В прошлом уроке мы жили с Chroma — «SQLite мира векторов». Сегодня делаем тот самый шаг, который бэкендеры делали десятки раз: **из embedded-базы в настоящий сервер**. Qdrant — это PostgreSQL нашей аналогии: отдельный процесс в Docker, HTTP/gRPC API, к которому ходят все ваши сервисы, фильтрация внутри HNSW-обхода (урок 2.2 — теперь вы знаете, почему это редкость и роскошь), снапшоты и путь к кластеру.

## 🎯 Цели урока

После этого урока вы сможете:

- поднимать Qdrant через **docker-compose** и понимать каждую строчку конфига;
- работать с **qdrant-client**: коллекции, точки (points), **payload** вместо метаданных;
- не наступить на два главных отличия от Chroma: **правила ID** и **score вместо distance**;
- строить фильтры `Filter`/`FieldCondition` и ускорять их **payload-индексами**;
- пользоваться персистентностью, **снапшотами** и Web UI;
- переносить коллекции с Chroma на Qdrant, не теряя качество (проверим eval'ом).

## План

1. Поднимаем сервер: docker-compose построчно
2. Клиент, коллекция, точки
3. Поиск: score, а не distance
4. Payload, фильтры и payload-индексы
5. Персистентность, снапшоты, Web UI
6. «Векторика» на Qdrant + eval

⏱ **Время:** ~75–90 минут.

> 💡 Понадобится установленный Docker (Desktop на Windows/macOS). Модель e5 — только для секции 6, как обычно с вежливым пропуском.

## 1. Поднимаем сервер: docker-compose построчно

В папке модуля лежит [`docker-compose.yml`](docker-compose.yml) — разберём его как конфиг, а не как заклинание:

```yaml
services:
  qdrant:
    image: qdrant/qdrant:v1.19.0   # версия ЗАКРЕПЛЕНА: мажор-минор сервера
    container_name: qdrant         # и python-клиента должны совпадать
    ports:
      - "6333:6333"   # REST API + Web UI — для нас и наших сервисов
      - "6334:6334"   # gRPC — быстрее для больших батчей, пригодится в проде
    volumes:
      - ./qdrant_storage:/qdrant/storage   # ДАННЫЕ ЖИВУТ ЗДЕСЬ, не в контейнере
    restart: unless-stopped        # пережить перезагрузку машины
```

Ключевая строчка — `volumes`: контейнер эфемерен, а данные лежат в папке `qdrant_storage` рядом с compose-файлом (bind mount). Удалите контейнер, обновите образ — данные останутся. Удалите папку — данных больше нет, и никакой Docker их не вернёт.

Команды, которые нужно знать:

```bash
docker compose up -d      # поднять в фоне (первый раз скачает образ, ~200 МБ)
docker compose ps         # статус
docker compose logs -f    # логи (Ctrl+C чтобы выйти)
docker compose restart    # перезапуск (данные живы)
docker compose down       # остановить и удалить контейнер (данные живы!)
```

После запуска откройте **http://localhost:6333/dashboard** — у Qdrant есть полноценный Web UI: коллекции, точки, консоль запросов. Держите его открытым весь урок.

In [ ]:
import httpx

QDRANT_URL = "http://localhost:6333"

try:
    info = httpx.get(QDRANT_URL, timeout=3).json()
    QDRANT_UP = True
    print(f"Qdrant отвечает: версия {info['version']}")
    print(f"Web UI: {QDRANT_URL}/dashboard")
except (httpx.ConnectError, httpx.TimeoutException):
    QDRANT_UP = False
    print("Qdrant не отвечает. Из папки модуля выполните в терминале:")
    print("  docker compose up -d")
    print("и перезапустите эту ячейку.")

In [ ]:
# Python-клиент. Держите его мажор-минор в пределах одной версии от сервера,
# иначе клиент честно предупредит о несовместимости.
%pip install -q qdrant-client

## 2. Клиент, коллекция, точки

Словарь меняется вслед за архитектурой: в Chroma были «записи в коллекции», в Qdrant — **точки** (points) — потому что это точки в векторном пространстве из урока 2.2. Схема же знакомая:

```text
точка (point)
├── id       1  или  "3f2b6c1e-..."   — int или UUID. СТРОКИ НЕЛЬЗЯ!
├── vector   [0.9, 0.02, 0.01, 0.02]
└── payload  {"doc_id": "cat", "topic": "животные", "meta": {"lang": "ru"}}
```

Два отличия от Chroma, на которых спотыкаются при миграции:

1. **ID — только целые числа или UUID.** Строковый slug вроде `"otpusk"` сервер отвергнет. Стандартный приём — детерминированный UUID из строки: `uuid.uuid5(NAMESPACE, "otpusk")` всегда даёт один и тот же id → переиндексация остаётся идемпотентной, а слаг переезжает в payload.
2. **Payload — полноценный JSON**, с вложенными объектами и списками (у Chroma — только плоский словарь). Фильтровать можно по вложенным ключам через точку: `meta.lang`.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

# timeout щедрый: массовые upsert'ы и снапшоты бывают неторопливы
client = QdrantClient(url=QDRANT_URL, timeout=60)

MINI = "mini"
if not client.collection_exists(MINI):          # идемпотентность вместо
    client.create_collection(                   # recreate_collection:
        collection_name=MINI,                   # в проде пересоздание = потеря данных
        vectors_config=VectorParams(size=4, distance=Distance.COSINE),
    )
print(f"Коллекция '{MINI}' готова:", client.get_collection(MINI).status)

Заметьте: метрика здесь — **обязательный параметр** (`Distance.COSINE`), забыть её невозможно. После граблей Chroma с дефолтным L2 это приятная строгость.

Заселяем ту же прозрачную мини-коллекцию из урока 2.3 — оси `[животные, техника, еда, спорт]`, чтобы результаты по-прежнему можно было проверять в уме:

In [ ]:
MINI_DOCS = [
    (1, "cat",    "Кот дремлет на подоконнике",              "животные", 4, [0.95, 0.02, 0.01, 0.02]),
    (2, "dog",    "Пёс с азартом гоняет мяч во дворе",       "животные", 7, [0.90, 0.02, 0.02, 0.35]),
    (3, "server", "Сервер перегрелся под нагрузкой",         "техника",  4, [0.02, 0.95, 0.01, 0.02]),
    (4, "db",     "База данных ждёт переиндексации",         "техника",  4, [0.05, 0.90, 0.02, 0.01]),
    (5, "pizza",  "Пицца с грибами уже остыла",              "еда",      5, [0.02, 0.02, 0.95, 0.05]),
    (6, "match",  "Финальный матч закончился со счётом 3:2", "спорт",    6, [0.10, 0.02, 0.05, 0.90]),
]

client.upsert(
    collection_name=MINI,
    points=[
        PointStruct(
            id=pid,
            vector=vector,
            payload={"doc_id": doc_id, "text": text, "topic": topic, "words": words},
        )
        for pid, doc_id, text, topic, words, vector in MINI_DOCS
    ],
)
print("Точек в коллекции:", client.count(MINI).count)

## 3. Поиск: score, а не distance

Единая точка входа для поиска в современном клиенте — `query_points`. И сразу — второе отличие от Chroma, источник самых тихих багов миграции:

In [ ]:
result = client.query_points(
    collection_name=MINI,
    query=[0.80, 0.0, 0.0, 0.40],   # тот же запрос "животные + спорт"
    limit=3,
)

for point in result.points:
    print(f"  score={point.score:.3f}  [{point.payload['doc_id']}] {point.payload['text']}")

Знакомая тройка `dog / cat / match` — но посмотрите на числа. Qdrant возвращает **`score` — похожесть**: больше = ближе (для косинуса это и есть cosine similarity). Chroma возвращала **`distances`**: меньше = ближе. Один и тот же поиск, противоположные конвенции:

| | Chroma | Qdrant |
|---|---|---|
| поле | `distances` | `score` |
| смысл | расстояние (1 − cos) | похожесть (cos) |
| ближе — это | **меньше** | **больше** |
| наш `dog` | 0.005 | 0.995 |

Мигрируете код между базами — первым делом проверьте пороги отсечения и сортировку: они инвертируются. Это не чья-то ошибка, это отсутствие стандарта в индустрии; ваша защита — тесты (eval!), а не память.

## 4. Payload, фильтры и payload-индексы

Фильтры в Qdrant — структурные объекты, а не словарики. Многословнее, чем в Chroma, зато композиция читается как SQL: `must` = AND, `should` = OR, `must_not` = NOT:

In [ ]:
from qdrant_client.models import FieldCondition, Filter, MatchAny, MatchValue, Range

query = [0.80, 0.0, 0.0, 0.40]

def show(label: str, query_filter: Filter | None) -> None:
    result = client.query_points(MINI, query=query, limit=4, query_filter=query_filter)
    found = [p.payload["doc_id"] for p in result.points]
    print(f"{label:<52} -> {found}")

show("без фильтра", None)

show('topic = "животные"',
     Filter(must=[FieldCondition(key="topic", match=MatchValue(value="животные"))]))

show("words >= 6",
     Filter(must=[FieldCondition(key="words", range=Range(gte=6))]))

show('topic IN (еда, спорт)',
     Filter(must=[FieldCondition(key="topic", match=MatchAny(any=["еда", "спорт"]))]))

show('животные И words >= 6',
     Filter(must=[
         FieldCondition(key="topic", match=MatchValue(value="животные")),
         FieldCondition(key="words", range=Range(gte=6)),
     ]))

show('НЕ техника',
     Filter(must_not=[FieldCondition(key="topic", match=MatchValue(value="техника"))]))

Вспомните урок 2.2 и «проклятие post-фильтра»: Qdrant фильтрует **во время обхода HNSW-графа** (filtered HNSW — та самая статья из ссылок урока 2.2). Выдача не опустошается, oversampling не нужен — фильтр просто делает часть вершин «невидимыми» для поиска.

Но на миллионах точек проверка условия на каждой вершине сама становится дорогой. Решение то же, что в реляционных БД: **индекс по полю**. Payload-индекс говорит Qdrant заранее разложить точки по значениям поля:

In [ ]:
# как CREATE INDEX в SQL: тип индекса = тип поля
client.create_payload_index(MINI, field_name="topic", field_schema="keyword")
client.create_payload_index(MINI, field_name="words", field_schema="integer")

# посчитать без поиска (задействует индекс)
from qdrant_client.models import FieldCondition, Filter, MatchValue
animals = client.count(
    MINI,
    count_filter=Filter(must=[FieldCondition(key="topic", match=MatchValue(value="животные"))]),
).count
print("Точек с topic=животные:", animals)

# scroll - постраничный обход коллекции (аналог SELECT ... LIMIT/OFFSET)
points, next_page = client.scroll(MINI, limit=3, with_vectors=False)
print("Первая страница:", [p.payload["doc_id"] for p in points])

info = client.get_collection(MINI)
print(f"Коллекция: {info.points_count} точек, статус {info.status}")

На шести точках индекс ничего не меняет — но привычка объявлять payload-индексы для всех полей, по которым фильтруете, спасёт вас на первом же миллионе. В Web UI (вкладка коллекции) видно, какие индексы объявлены.

## 5. Персистентность, снапшоты, Web UI

**Персистентность** уже работает — молча. Всё, что вы записали, лежит в `qdrant_storage/` рядом с compose-файлом. Проверьте: `docker compose restart` — и коллекция на месте. Пара правил гигиены:

- `qdrant_storage/` — **бинарные данные БД**: не коммитьте в git (добавьте в `.gitignore`), не редактируйте руками;
- бэкап «скопировать папку» работает только на остановленном сервере; для живого — снапшоты.

**Снапшоты** — консистентный бэкап коллекции через API: `client.create_snapshot(...)`. Файл создаётся внутри volume, скачивается по HTTP, восстанавливается на любом другом сервере Qdrant.

> ⚠️ Честное предупреждение: на Windows с bind mount снапшот даже крошечной коллекции занимает десятки секунд — Docker Desktop медленно гоняет файлы через границу WSL2. На Linux-сервере это секунды. Не пугайтесь в упражнении 3.

**CRUD-остатки** — всё, что положено серверной БД:

In [ ]:
# READ по id
point = client.retrieve(MINI, ids=[5], with_vectors=False)[0]
print("retrieve(5):", point.payload["text"])

# UPDATE payload БЕЗ перезаписи вектора: set_payload обновляет только
# указанные ключи (полная замена - overwrite_payload)
client.set_payload(MINI, payload={"words": 6, "edited": True}, points=[5])
print("после set_payload:", client.retrieve(MINI, ids=[5], with_vectors=False)[0].payload)

# DELETE точек по id
client.delete(MINI, points_selector=[6])
print("после delete(6):", client.count(MINI).count, "точек")

`set_payload` — то, чего не хватало в Chroma: обновить метаданные, **не трогая вектор** и не пересчитывая эмбеддинг. Для сценариев вроде «пометить документ устаревшим» или «сменить владельца» — ровно то, что нужно (и основа упражнения 2).

## 6. «Векторика» на Qdrant + eval

Переносим базу знаний. Всё как в уроке 2.3, плюс трюк с детерминированными UUID вместо строковых слагов:

In [ ]:
import numpy as np

EMB_MODEL_NAME = "intfloat/multilingual-e5-small"

model = None
try:
    from sentence_transformers import SentenceTransformer
    from huggingface_hub import snapshot_download
    snapshot_download(EMB_MODEL_NAME, local_files_only=True)   # только из кэша
    model = SentenceTransformer(EMB_MODEL_NAME)
    print("Модель загружена — секция будет живой.")
except Exception:
    print("Модель e5 не найдена в кэше (установка — в уроке 2.1); секция пропустится.")

EMB_AVAILABLE = model is not None

def embed_texts(texts: list[str], *, prefix: str) -> np.ndarray:
    """Эмбеддинги с обязательным для e5 префиксом (урок 2.1)."""
    return model.encode([f"{prefix}: {t}" for t in texts], normalize_embeddings=True)

In [ ]:
import uuid

from utils import load_documents

docs = load_documents("data")
KB = "vectorika"

def slug_to_id(slug: str) -> str:
    """Детерминированный UUID из слага: 'otpusk' -> всегда один и тот же id."""
    return str(uuid.uuid5(uuid.NAMESPACE_URL, slug))

if QDRANT_UP and EMB_AVAILABLE:
    if not client.collection_exists(KB):
        client.create_collection(
            KB, vectors_config=VectorParams(size=384, distance=Distance.COSINE))
    client.create_payload_index(KB, field_name="topic", field_schema="keyword")

    vectors = embed_texts([d.text for d in docs], prefix="passage")
    client.upsert(KB, points=[
        PointStruct(
            id=slug_to_id(d.doc_id),
            vector=vectors[i].tolist(),
            payload={"doc_id": d.doc_id, "topic": d.topic,
                     "title": d.title, "text": d.text},
        )
        for i, d in enumerate(docs)
    ])
    print(f"Проиндексировано {client.count(KB).count} документов\n")

    def search(question: str, where: Filter | None = None) -> None:
        q_vec = embed_texts([question], prefix="query")[0]
        res = client.query_points(KB, query=q_vec.tolist(), limit=3, query_filter=where)
        print(f"Вопрос: {question}")
        for p in res.points:
            print(f"  score={p.score:.3f}  {p.payload['doc_id']:<14} {p.payload['title']}")
        print()

    search("Как оформить отгул на пару дней?")
    search("Сломался рабочий ноутбук, куда бежать?",
           Filter(must=[FieldCondition(key="topic", match=MatchValue(value="it"))]))
else:
    print("[пропущено: нужны запущенный Qdrant и модель e5]")

In [ ]:
EVAL_QUESTIONS = [
    ("Сколько дней отпуска положено в году?",              "otpusk"),
    ("Как оформить больничный лист?",                      "bolnichny"),
    ("VPN подключён, но внутренние сайты не открываются",  "vpn"),
    ("Кто должен посмотреть мой код перед мержем?",        "code-review"),
    ("Упал прод, что делать в первую очередь?",            "incidents"),
]

if QDRANT_UP and EMB_AVAILABLE:
    top1 = top3 = 0
    for question, expected in EVAL_QUESTIONS:
        q_vec = embed_texts([question], prefix="query")[0]
        res = client.query_points(KB, query=q_vec.tolist(), limit=3)
        top_ids = [p.payload["doc_id"] for p in res.points]
        top1 += top_ids[0] == expected
        top3 += expected in top_ids
    total = len(EVAL_QUESTIONS)
    print(f"top-1 accuracy: {top1}/{total}   top-3 hit rate: {top3}/{total}")
    print("Ожидаемо совпадает с уроками 2.1 и 2.3: модель и метрика те же.")
else:
    print("[пропущено: нужны запущенный Qdrant и модель e5]")

## Что мы получили за Docker

Итоговое сравнение — теперь из личного опыта, а не из маркетинга:

| | Chroma (урок 2.3) | Qdrant (сегодня) |
|---|---|---|
| архитектура | библиотека в процессе | сервер, HTTP + gRPC |
| клиентов одновременно | один процесс | сколько угодно сервисов |
| ID | любые строки | int / UUID (слаг → `uuid5`) |
| результат поиска | `distances` (меньше=ближе) | `score` (больше=ближе) |
| метаданные | плоский словарь | вложенный JSON + payload-индексы |
| фильтры | есть | внутри HNSW-обхода, масштабируются |
| обновление payload | только с векторм (upsert) | `set_payload` без вектора |
| бэкапы | «скопируй папку» | снапшоты по API на живом сервере |
| дальше по масштабу | — | квантизация, шардирование, кластер |
| цена | ноль | Docker + обычный серверный ops (урок 1.3!) |

Правило выбора не изменилось с урока 2.2: Chroma — пока векторный поиск «внутри» одного приложения; Qdrant — как только он становится **сервисом**: несколько клиентов, миллионы точек, фильтры на масштабе, требования к бэкапам.

## ⚠️ Частые ошибки

1. **Строковые ID.** `id="otpusk"` — ошибка сервера. Только int или UUID; слаг кладите в payload, id делайте через `uuid.uuid5` — детерминированно и идемпотентно.
2. **Скопировать пороги из Chroma.** Там 0.2 значило «очень близко» (distance), тут — «очень далеко» (score). Инвертированные конвенции тихо ломают отсечение мусора. Защита — eval после миграции.
3. **Фильтры без payload-индексов.** На демо незаметно, на миллионе точек — деградация фильтрованных запросов. Объявляйте индекс на каждое поле из фильтров.
4. **`qdrant_storage/` в git.** Бинарные файлы БД раздуют репозиторий мгновенно. В `.gitignore` — сразу при создании проекта.
5. **`recreate_collection` в проде.** Пересоздание = потеря всех точек. Идемпотентный паттерн: `collection_exists` → `create_collection`, данные — через `upsert`.
6. **Считать, что данные живут в контейнере.** `docker compose down` безопасен (volume остаётся), а вот удаление папки `qdrant_storage` — это удаление базы. Куда смотрит volume — первое, что проверяют перед экспериментами с контейнером.

## 🏋️ Упражнения

### Упражнение 1 — фильтры-конструктор 🧩

Соберите один запрос к мини-коллекции: похожее на «животные», из тем «животные» ИЛИ «еда» (`should`), но НЕ короче 5 слов (`must` + `Range`) и НЕ пицца (`must_not` по `doc_id`). Предскажите выдачу до запуска.

### Упражнение 2 — жизненный цикл документа 🔄

На коллекции «Векторики» (или мини): пометьте документ `otpusk` устаревшим через `set_payload` (`{"status": "deprecated"}`), убедитесь, что вектор не изменился (`retrieve` с `with_vectors=True` до и после), и напишите поиск, который исключает устаревшие документы фильтром. Это продакшен-паттерн «мягкого удаления».

### Упражнение 3 — бэкап по-взрослому 📦

Создайте снапшот мини-коллекции. Нюанс: на Windows bind mount операция занимает до минуты и синхронный вызов может упереться в таймаут клиента — используйте `wait=False` и поллинг `list_snapshots` до готовности (типовой паттерн тяжёлых операций БД). Посмотрите на результат в Web UI (Collections → Snapshots) и опишите словами сценарий восстановления на новом сервере.

## 📝 Мини-квиз

**1. Почему `id="otpusk"` не работает и как правильно?**

<details><summary>Ответ</summary>

Qdrant принимает только целые числа и UUID как id точек. Правильный паттерн: `uuid.uuid5(NAMESPACE, "otpusk")` — детерминированный UUID (один и тот же при каждом вызове), а человекочитаемый слаг — в payload. Идемпотентность переиндексации сохраняется.

</details>

**2. После миграции с Chroma поиск «отсекает мусор» порогом 0.3. Что сломалось?**

<details><summary>Ответ</summary>

Конвенция инвертировалась: в Chroma 0.3 — это distance (довольно близко), в Qdrant — score (довольно далеко). Порог, отсекавший мусор, теперь отсекает лучшие результаты. Пороги и сортировки пересматриваются при любой смене БД, проверка — eval.

</details>

**3. Зачем payload-индексы, если фильтры работают и без них?**

<details><summary>Ответ</summary>

Без индекса условие проверяется «на лету» на каждой посещённой вершине графа — на больших коллекциях это медленно. Индекс раскладывает точки по значениям поля заранее — как B-tree индекс в SQL. Правило: индекс на каждое поле, участвующее в фильтрах.

</details>

**4. Чем фильтрация Qdrant отличается от post-фильтра из урока 2.2?**

<details><summary>Ответ</summary>

Post-filter отбрасывает неподходящее *после* поиска топ-k — выдача может опустеть. Qdrant применяет фильтр *во время* обхода HNSW-графа: неподходящие вершины невидимы для поиска, топ-k всегда полон (если подходящих точек хватает), oversampling не нужен.

</details>

**5. `docker compose down`, обновление образа, `up -d`. Что с данными?**

<details><summary>Ответ</summary>

Живы: они в bind-mount папке `qdrant_storage`, а не в контейнере. Контейнеры эфемерны и заменяемы; данные привязаны к volume. Опасно другое — удалить саму папку или перепутать путь в `volumes`.

</details>

## ✅ Решения упражнений

### Решение 1 — фильтры-конструктор

In [ ]:
combo = Filter(
    should=[   # хотя бы одно из
        FieldCondition(key="topic", match=MatchValue(value="животные")),
        FieldCondition(key="topic", match=MatchValue(value="еда")),
    ],
    must=[FieldCondition(key="words", range=Range(gte=5))],
    must_not=[FieldCondition(key="doc_id", match=MatchValue(value="pizza"))],
)

result = client.query_points(MINI, query=[0.9, 0.0, 0.0, 0.1],
                             limit=4, query_filter=combo)
for p in result.points:
    print(f"  score={p.score:.3f}  {p.payload['doc_id']}  ({p.payload['topic']}, "
          f"{p.payload['words']} слов)")

Ожидание: остаётся только `dog` — кот из «животных» отсеян длиной (4 слова), пицца исключена явно (к тому же после секции CRUD у неё 6 слов — но `must_not` сильнее), матч удалён ещё в секции CRUD. Комбинируя `should`/`must`/`must_not`, вы собираете любую булеву логику — как WHERE с AND/OR/NOT.

### Решение 2 — жизненный цикл документа

In [ ]:
if QDRANT_UP and EMB_AVAILABLE:
    target = slug_to_id("otpusk")

    vector_before = client.retrieve(KB, ids=[target], with_vectors=True)[0].vector
    client.set_payload(KB, payload={"status": "deprecated"}, points=[target])
    after = client.retrieve(KB, ids=[target], with_vectors=True)[0]

    print("Вектор не изменился:", np.allclose(vector_before, after.vector))
    print("Payload:", {k: after.payload[k] for k in ("doc_id", "status")})

    q_vec = embed_texts(["Сколько дней отпуска положено?"], prefix="query")[0]
    res = client.query_points(
        KB, query=q_vec.tolist(), limit=3,
        query_filter=Filter(must_not=[
            FieldCondition(key="status", match=MatchValue(value="deprecated"))]),
    )
    print("Поиск без устаревших:", [p.payload["doc_id"] for p in res.points])

    # вернём как было, чтобы ноутбук оставался идемпотентным
    client.set_payload(KB, payload={"status": "active"}, points=[target])
else:
    print("[пропущено: нужны запущенный Qdrant и модель e5]")

Обратите внимание: документ про отпуск исчез из выдачи по «отпускному» запросу — фильтр работает, вектор нетронут, эмбеддинг пересчитывать не пришлось. «Мягкое удаление», версионирование, права доступа — всё это в Qdrant делается payload'ом + фильтром, без переиндексации.

### Решение 3 — бэкап по-взрослому

In [ ]:
import time

if QDRANT_UP:
    # приборка от прошлых запусков ячейки
    for old in client.list_snapshots(MINI):
        client.delete_snapshot(MINI, old.name)

    # wait=False: сервер начинает работу и сразу отвечает, не держа соединение.
    # Классический паттерн для тяжёлых операций БД: запустил - поллишь готовность.
    client.create_snapshot(collection_name=MINI, wait=False)
    print("Снапшот строится в фоне, опрашиваем готовность...")

    snapshot = None
    for attempt in range(30):                # ждём до двух минут
        time.sleep(4)
        ready = client.list_snapshots(MINI)
        if ready:
            snapshot = ready[-1]
            break
        print(f"  ...ещё строится ({(attempt + 1) * 4} с)")

    if snapshot:
        print(f"Готов: {snapshot.name}  ({snapshot.size / 2**20:.1f} МБ)")
        client.delete_snapshot(MINI, snapshot.name)   # учебный - удаляем
        print("Учебный снапшот удалён.")
    else:
        print("За 2 минуты не построился — проверьте позже: client.list_snapshots(MINI)")
else:
    print("[пропущено: Qdrant не запущен]")

Сценарий восстановления: скачать `.snapshot`-файл по HTTP (`GET /collections/mini/snapshots/<имя>`), перенести на новый сервер и восстановить (`PUT /collections/mini/snapshots/upload`) — коллекция поднимется со всеми точками, payload'ами и индексами. Это и бэкап, и способ переезда между серверами, и «золотой образ» для стендов.

## 🎓 Итоги

- Qdrant — это «PostgreSQL после SQLite»: отдельный сервер в Docker, данные в volume (`qdrant_storage/`), Web UI на 6333.
- Точки: id — **int/UUID** (слаг → `uuid5`), payload — полноценный JSON. Поиск возвращает **score = похожесть** — противоположность Chroma, пороги при миграции пересматриваются.
- Фильтры (`must`/`should`/`must_not`) работают **внутри** HNSW-обхода; на масштабе им нужны **payload-индексы** — как WHERE нужен CREATE INDEX.
- `set_payload` меняет метаданные без пересчёта векторов — основа мягкого удаления и версионирования. Снапшоты — консистентный бэкап по API.
- Eval подтвердил: миграция без потерь. Одна модель + одна метрика = одинаковое качество на numpy, Chroma и Qdrant; меняется только инженерная обвязка.

**Дальше — урок 2.5 «Полезные практики: чанкинг, метаданные, reranking»:** пока мы индексировали документы целиком — и вектор «обо всём» размывает поиск. Научимся резать тексты правильно и пересортировывать выдачу.

## 📚 Что почитать

- [Быстрый старт Qdrant](https://qdrant.tech/documentation/quickstart/) — официальный тур
- [Фильтрация](https://qdrant.tech/documentation/concepts/filtering/) — вся мощь Filter/FieldCondition
- [Индексирование payload](https://qdrant.tech/documentation/concepts/indexing/) — payload-индексы и их типы
- [Снапшоты](https://qdrant.tech/documentation/concepts/snapshots/) — бэкапы и восстановление
- [Filtrable HNSW](https://qdrant.tech/articles/filtrable-hnsw/) — как устроена фильтрация в графе (знакомо после урока 2.2)